In [ ]:
# Remember to make a venv and install the dependencies before running this notebook. You can do this with pip install -r requirements.txt if you have a requirements.txt file with the necessary dependencies, 
# py -m venv venv && source venv/bin/activate && pip install -r requirements.txt

# Or you can run the following commands to install them directly (after making your venv and activating it):

!pip install -q pandas numpy torch pathlib kaggle
!pip install git+https://github.com/unslothai/unsloth-zoo.git git+https://github.com/unslothai/unsloth.git trl==0.24.0 bitsandbytes picosvg cairosvg opencv-python-headless editdistance mergekit llm_blender outlines scikit-learn weave wandb protobuf llguidance

# !pip install -r requirements.txt

  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-req-build-oj_t2o6c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-req-build-oj_t2o6c
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit e5944f23827b49eda601e6168fa17def8338e96f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-6jt1m1co
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-6jt1m1co
  Resolved https://github.com/unslothai/unsloth.git to commit 362ad3606b845c51ae5bf2354ff84d14e6c3356b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth_zoo: filename=unsloth_zoo-2026.3.6-py3-none-any.whl size=400756 sha256=e

In [2]:
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("./dl-spring-2026-svg-generation")

# Download data
if not BASE_DIR.exists() or not any(BASE_DIR.iterdir()):
    print("Downloading and unzipping data...")
    os.makedirs(BASE_DIR, exist_ok=True)
    print("Downloading...")
    !kaggle competitions download -c dl-spring-2026-svg-generation-from-text-prompts-extended-deadline --force
    print("Unzipping...")
    !unzip -q dl-spring-2026-svg-generation.zip
    !mv sample_submission.csv dl-spring-2026-svg-generation
    !mv train.csv dl-spring-2026-svg-generation
    !mv test.csv dl-spring-2026-svg-generation
else:
    print("SVG Data folder already exists.")

SVG Data folder already exists.


In [4]:
import pandas as pd
import re

print("Analyzing SVG Dimensions in the Training Set.")

# Load the dataset
df = pd.read_csv("./dl-spring-2026-svg-generation/train.csv")

# Regex to safely find attributes inside the opening <svg ... > tag
height_pattern = re.compile(r'<svg[^>]*\sheight="([^"]+)"')
width_pattern = re.compile(r'<svg[^>]*\swidth="([^"]+)"')
viewbox_pattern = re.compile(r'<svg[^>]*\sviewBox="([^"]+)"')

heights = []
widths = []
viewboxes = []

# Scan the dataset
for svg in df['svg'].dropna():
    # We only search the first 250 characters to save time (the opening tag)
    head = svg[:250] 
    
    h_match = height_pattern.search(head)
    w_match = width_pattern.search(head)
    vb_match = viewbox_pattern.search(head)
    
    if h_match: heights.append(h_match.group(1))
    if w_match: widths.append(w_match.group(1))
    if vb_match: viewboxes.append(vb_match.group(1))

# Print the Results
print("\n--- Top 5 Heights ---")
print(pd.Series(heights).value_counts().head(5).to_string())

print("\n--- Top 5 Widths ---")
print(pd.Series(widths).value_counts().head(5).to_string())

print("\n--- Top 5 ViewBoxes ---")
print(pd.Series(viewboxes).value_counts().head(5).to_string())

total_svgs = len(df)
print(f"\nTotal SVGs scanned: {total_svgs}")

Analyzing SVG Dimensions in the Training Set.

--- Top 5 Heights ---
200.0px    40892
128         6308
400          740
200          394
100          245

--- Top 5 Widths ---
200.0px    40892
128         6308
400          733
200          384
100          245

--- Top 5 ViewBoxes ---
0.0 0.0 200.0 200.0    40892
0 0 24 24               2104
0 0 128 128              902
0 0 400 400              734
0 0 32 32                655

Total SVGs scanned: 50000


In [ ]:
import pandas as pd
import numpy as np
import concurrent.futures
import re
import os
from picosvg.svg import SVG

# Regex Compilations
FLOAT_PATTERN = re.compile(r'-?\d+\.\d+')
PATH_LETTER_SPACE_PATTERN = re.compile(r'([a-zA-Z])\s+')
COMMA_SPACE_PATTERN = re.compile(r'\s*,\s*')
HEX_COLOR_PATTERN = re.compile(r'#([0-9a-fA-F])\1([0-9a-fA-F])\2([0-9a-fA-F])\3')
MULTIPLE_SPACES_PATTERN = re.compile(r'\s+')
PATH_D_ATTR_PATTERN = re.compile(r'd="([^"]+)"')

# Cleaning Helper Functions 

def round_floats(match):
    # Strictly maintains 2 decimal points, but strips useless trailing zeros
    return f"{float(match.group(0)):.2f}".rstrip('0').rstrip('.')

def minify_d_attr(match):
    """Cleans spaces strictly inside the path string."""
    d_str = match.group(1)
    # Remove spaces after path command letters (e.g., "M 10" -> "M10")
    d_str = PATH_LETTER_SPACE_PATTERN.sub(r'\1', d_str)
    # Remove spaces around commas (e.g., "10, 20" -> "10,20")
    d_str = COMMA_SPACE_PATTERN.sub(',', d_str)
    # Condense multiple spaces into one
    d_str = MULTIPLE_SPACES_PATTERN.sub(' ', d_str)
    return f'd="{d_str}"'

def minify_path_and_hex(svg_string):
    # Apply the aggressive space removal only to the d="..." attribute
    svg_string = PATH_D_ATTR_PATTERN.sub(minify_d_attr, svg_string)
    
    # Compress hex colors globally (e.g., #FFFFFF -> #FFF)
    svg_string = HEX_COLOR_PATTERN.sub(r'#\1\2\3', svg_string)
    
    # Condense global multiple spaces safely
    svg_string = MULTIPLE_SPACES_PATTERN.sub(' ', svg_string)
    
    return svg_string

def strip_useless_tags(svg_string):
    # Remove empty defs
    svg_string = svg_string.replace("<defs/>", "").replace("<defs></defs>", "")
    
    # Remove redundant/default attributes to save tokens
    svg_string = re.sub(r'\s*fill-opacity="1\.0"', '', svg_string)
    svg_string = re.sub(r'\s*filling="0"', '', svg_string)
    svg_string = re.sub(r'\s*fill=""', '', svg_string)
    
    # Normalize height and width instead of deleting them
    # This changes height="200.0px" or width="256px" to simply height="200"
    svg_string = re.sub(r'\s*height="[0-9.]+(px)?"', ' height="200"', svg_string)
    svg_string = re.sub(r'\s*width="[0-9.]+(px)?"', ' width="200"', svg_string)
    
    # Close gaps between XML tags
    svg_string = svg_string.replace('> <', '><')
    
    return svg_string

# Main Optimization Wrapper 
def optimize_svg(raw_svg):
    if not isinstance(raw_svg, str) or not raw_svg.strip(): 
        return ""
        
    try:
        # Step A: Standardize shapes to paths using picosvg
        svg_obj = SVG.fromstring(raw_svg)
        cleaned_svg = svg_obj.topicosvg().tostring()
        
        # Step B: Apply 2-decimal float rounding
        cleaned_svg = FLOAT_PATTERN.sub(round_floats, cleaned_svg)
        
        # Step C: Minify paths and compress hex codes
        cleaned_svg = minify_path_and_hex(cleaned_svg)
        
        # Step D: Strip useless XML bloat
        cleaned_svg = strip_useless_tags(cleaned_svg)
        
        return cleaned_svg.strip()
    except Exception:
        return ""

# Parallel Processing Execution 
def process_chunk(chunk):
    chunk['svg'] = chunk['svg'].apply(optimize_svg)
    return chunk
    
# This process took 20 minutes with 4 cores (Comment out the below lines if you already have the train_optimized.parquet file)
if __name__ == "__main__":
    print("Starting Data Diet.")
    
    df = pd.read_csv("./dl-spring-2026-svg-generation/train.csv")
    
    num_cores = os.cpu_count() or 4
    num_cores = 4
    print(f"At least {num_cores} Cores are available.")

    indices = np.array_split(df.index, num_cores * 4)
    chunks = [df.loc[idx].copy() for idx in indices]

    processed_chunks = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
        for result in executor.map(process_chunk, chunks):
            processed_chunks = list(executor.map(process_chunk, chunks))

    final_df = pd.concat(processed_chunks)
    
    # Filter out any SVGs that failed parsing or returned empty
    final_df = final_df[final_df['svg'] != ""]
    
    final_df.to_parquet("./train_optimized.parquet", index=False)
    print(f"Data Diet Complete. {len(final_df)} highly compressed rows saved.")

In [6]:
# Load your newly compressed dataset
df = pd.read_parquet("./train_optimized.parquet")

print(f"Total optimized rows ready for training: {len(df)}\n")
print("==================================================")
print("       FIRST 10 SVGS FOR SANITY CHECK")
print("==================================================\n")

# Loop through the first 10 rows
for i, row in df.head(10).iterrows():
    print(f"[{i+1}/10] PROMPT: {row['prompt']}")
    print(f"SVG STRING:\n{row['svg']}\n")
    print("-" * 80 + "\n")

Total optimized rows ready for training: 49043

       FIRST 10 SVGS FOR SANITY CHECK

[1/10] PROMPT: The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background.
SVG STRING:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" width="200"><path fill="#FF6A00" d="M93.3,21.2 L93.3,80.4 L21.2,80.4 L21.2,179.6 L120.4,179.6 L120.4,107.1 L179.1,107.1 L179.1,21.2 L93.3,21.2 ZM113.8,172.9 L27.9,172.9 L27.9,87.1 L113.7,87.1 L113.7,172.9 L113.8,172.9 ZM172.5,100.4 L120.4,100.4 L120.4,80.4 L100,80.4 L100,27.9 L172.5,27.9 L172.5,100.4 Z"/><path fill="#FF6A00" d="M143.8,44.2 C143.8,40 140.5,37.1 136.7,37.1 C132.5,37.1 129.6,40.4 129.6,44.2 L129.6,51.3 L144.2,51.3 L144.2,44.2 L143.8,44.2 ZM129.2,62.1 C129.2,66.3 132.5,69.2 136.3,69.2 C140.5,69.2 143.4,65.9 143.4,62.1 L143.4,55 L129.2,55 L129.2,62.1 Z"/><path fill="#FF6A00" d="M149.6,62.1 C149.6,68.8 144.6,74.2 138.4,75 L135,75 C128.8,74.2 123.8,68.8 123.8,62.1 L116.

### Training

In [7]:
# import os
# import torch
# from unsloth import FastLanguageModel

# import transformers.utils.hub
# transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", "~/.cache/huggingface/hub")
# os.environ["WANDB_DISABLED"] = "true"

# from trl import SFTTrainer, SFTConfig
# from datasets import load_dataset

# if __name__ == "__main__":
#     train_optimized_filepath = "./train_optimized.parquet"
#     raw_dataset = load_dataset("parquet", data_files=train_optimized_filepath, split="train")
    
#     # --- TWEAK: Split into Train and Eval ---
#     dataset_split = raw_dataset.select(range(25000)).train_test_split(test_size=500, seed=25)
#     train_dataset = dataset_split["train"]
#     eval_dataset = dataset_split["test"]
    
#     MODEL_ID = "unsloth/Qwen2.5-Coder-3B"
    
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name=MODEL_ID,
#         max_seq_length=2048,
#         dtype=torch.float16, 
#         load_in_4bit=True,
#     )
    
#     model = FastLanguageModel.get_peft_model(
#         model, 
#         # --- TWEAK: Increased Capacity & MLP Layers ---
#         r=32, 
#         lora_alpha=64, 
#         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
#         lora_dropout=0, 
#         bias="none",
#         use_gradient_checkpointing=True,
#         random_state=25,
#     )
    
#     EOS = tokenizer.eos_token
    
#     def format_for_sft(example):
#         open_comment = "<!-" + "-"
#         close_comment = "-" + "->"
#         text = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n{example['svg']}\n```{EOS}"
#         return {"text": text}
    
#     train_dataset = train_dataset.map(format_for_sft)
#     eval_dataset = eval_dataset.map(format_for_sft)
    
#     sft_config = SFTConfig(
#         output_dir="./svg-phase1-sft",
#         dataset_text_field="text",
#         max_seq_length=2048,
        
#         # --- TWEAK: Dataset Packing ---
#         packing=True, 
        
#         num_train_epochs=1, 
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=8, 
        
#         # --- TWEAK: Advanced Training Dynamics ---
#         learning_rate=2e-4,
#         lr_scheduler_type="cosine", 
#         warmup_steps=100,           
#         weight_decay=0.01,          
#         neftune_noise_alpha=5,      
        
#         fp16=True,  
#         bf16=False, 
#         optim="paged_adamw_8bit", 
#         logging_steps=50,
        
#         # --- TWEAK: Evaluation Strategy ---
#         eval_strategy="steps",
#         eval_steps=200, 
        
#         # --- NEW: Save exactly when you evaluate, and keep the best one! ---
#         save_strategy="steps", 
#         save_steps=200,
#         load_best_model_at_end=True,         # Auto-reverts to the best checkpoint!
#         metric_for_best_model="eval_loss",   # Uses your eval set as the judge
#         greater_is_better=False,             # Because lower loss is better
#         save_total_limit=2,
#         ddp_find_unused_parameters=False,
#     )
    
#     sft_trainer = SFTTrainer(
#         model=model, 
#         tokenizer=tokenizer, 
#         train_dataset=train_dataset, 
#         eval_dataset=eval_dataset, # Hooked up the eval set
#         args=sft_config
#     )
    
#     print("Starting SFT Phase 1 with Qwen2.5-Coder-3B...")
#     sft_trainer.train()
    
#     model.save_pretrained("./sft-lora-adapter-3B")
#     tokenizer.save_pretrained("./sft-lora-adapter-3B")

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model to CPU.")
# 1. Load the base model strictly to the CPU to avoid the 8GB VRAM limit
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-Coder-3B", 
    torch_dtype=torch.float16,
    device_map="cpu", 
)

print("Attaching LoRA adapter.")
# 2. Attach your Phase 1 adapter
model = PeftModel.from_pretrained(
    base_model,
    "./sft-lora-adapter-3B",
    device_map="cpu",
)

print("Merging weights (this will take a while).")
# 3. Fuse the LoRA weights directly into the base model's linear layers
merged_model = model.merge_and_unload()

print("Saving merged model to disk.")
# 4. Export the standalone Hugging Face model
merged_model.save_pretrained("./sft-merged-model-3b")

tokenizer = AutoTokenizer.from_pretrained("./sft-lora-adapter-3B")
# tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/jef9921/sft-lora-3b/sft-lora-adapter-3B")
tokenizer.save_pretrained("./sft-merged-model-3b")

print("Merge complete! Safe for Outlines inference/GRPO.")

/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading base model to CPU.


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:32<00:00, 13.19it/s]


Attaching LoRA adapter.
Merging weights (this will take a while).
Saving merged model to disk.


Writing model shards: 100%|██████████| 1/1 [03:42<00:00, 222.18s/it]


Merge complete! Safe for Outlines inference/GRPO.


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Load your local SFT-trained tokenizer
# You can point this to "./sft-merged-model-3b" or "Qwen/Qwen2.5-Coder-3B"
TOKENIZER_PATH = "./sft-merged-model-3b" 

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# Load the highly optimized dataset
PARQUET_PATH = "./train_optimized.parquet"
print(f"Loading {PARQUET_PATH}...")
df = pd.read_parquet(PARQUET_PATH)

print("Calculating exact token lengths for 49,043 SVGs (this takes about 60 seconds)...")

# Encode the SVGs and count the tokens
# We add_special_tokens=False because we only care about the SVG payload length
lengths = df['svg'].apply(lambda x: len(tokenizer.encode(str(x), add_special_tokens=False)))

# Extract the critical metrics
max_len = lengths.max()
p99_len = lengths.quantile(0.99)
p95_len = lengths.quantile(0.95)
avg_len = lengths.mean()

print("\n======================================")
print("       SVG TOKEN LENGTH STATS         ")
print("======================================")
print(f"Absolute Max Length: {max_len} tokens")
print(f"99th Percentile:     {p99_len:.0f} tokens")
print(f"95th Percentile:     {p95_len:.0f} tokens")
print(f"Average Length:      {avg_len:.0f} tokens")
print("======================================")

Loading tokenizer...
Loading ./train_optimized.parquet...
Calculating exact token lengths for 49,043 SVGs (this takes about 60 seconds)...


Token indices sequence length is longer than the specified maximum sequence length for this model (33864 > 32768). Running this sequence through the model will result in indexing errors



       SVG TOKEN LENGTH STATS         
Absolute Max Length: 44909 tokens
99th Percentile:     4988 tokens
95th Percentile:     2503 tokens
Average Length:      1173 tokens


In [9]:
import gc

del model, base_model, merged_model, tokenizer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import os
import pandas as pd
import torch
import io
import cairosvg
import numpy as np
from PIL import Image
import xml.etree.ElementTree as ET
import warnings
import re

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel
import outlines
from outlines.types import CFG

warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

print("Initializing local inference.")
device = "cuda"

# ==========================================
# LOAD MAIN MODEL (16-BIT PRECISION)
# ==========================================
print("Loading 3B model in 16-bit (Float16).")
hf_model = AutoModelForCausalLM.from_pretrained(
    "./sft-merged-model-3b", 
    torch_dtype=torch.float16, # This replaces the 4-bit quantization!
    device_map=device, 
)

hf_model.generation_config.max_length = None
hf_tokenizer = AutoTokenizer.from_pretrained("./sft-merged-model-3b")

# The Outlines CFG guarantees 100% valid SVG formatting
generator = outlines.from_transformers(hf_model, hf_tokenizer)

official_kaggle_grammar = CFG("""
    ?start: WS? svg WS?
    svg: "<svg" WS? ATTR_LIST ">" WS? elements "</svg>"
    elements: (element | TEXT)*
    element: "<" TAG WS? ATTR_LIST "/>" WS? | "<" TAG WS? ATTR_LIST ">" WS? elements "</" TAG ">" WS?
    TAG: "svg" | "g" | "path" | "rect" | "circle" | "ellipse" | "line" | "polyline" | "polygon" | "defs" | "use" | "symbol" | "clipPath" | "mask" | "linearGradient" | "radialGradient" | "stop" | "text" | "tspan" | "title" | "desc" | "style" | "pattern" | "marker" | "filter"
    ATTR_LIST: (/[a-zA-Z0-9_:-]+/ WS? "=" WS? /"[^"]*"/ WS?)*
    TEXT: /[^<]+/
    WS: /[ \\t\\n\\r]+/
""")

# ==========================================
# 2. LOAD SIGLIP JUDGE (16-BIT PRECISION)
# ==========================================
print("Loading SigLIP Judge.")
siglip_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(siglip_id)

judge = AutoModel.from_pretrained(
    siglip_id, 
    torch_dtype=torch.float16 # Crucial for saving VRAM
).to(device)
judge.eval() 


/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!


Initializing local inference.
Loading 3B model in 16-bit (Float16).


Loading weights: 100%|██████████| 434/434 [01:10<00:00,  6.18it/s]


Loading SigLIP Judge.


The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 888/888 [00:19<00:00, 45.32it/s]


SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 1152)
      (position_embedding): Embedding(64, 1152)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-26): 27 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (fc2): Linear(in_features=4304,

In [ ]:
# ==========================================
# HELPER FUNCTIONS
# ==========================================
def extract_svg(text):
    match = re.search(r'(<svg.*?</svg>)', text, re.IGNORECASE | re.DOTALL)
    return match.group(1) if match else text


def heal_svg(raw_svg):
    """Saves truncated SVGs by chopping off broken paths and closing the tag."""
    if raw_svg.strip().endswith("</svg>"):
        return raw_svg
        
    # Find the last perfectly closed element (like a completed <path />)
    last_closed_idx = raw_svg.rfind("/>")
    
    if last_closed_idx != -1:
        # Chop off the broken half-written path and seal the SVG
        healed_svg = raw_svg[:last_closed_idx + 2] + "\n</svg>"
        return healed_svg
        
    return raw_svg

def is_kaggle_compliant(svg_string):
    if len(svg_string) > 16000: return False
    try:
        root = ET.fromstring(svg_string)
        if root.tag.split('}')[-1] != 'svg': return False
    except ET.ParseError: return False
    if svg_string.count("<path") > 256: return False
    return True

def render_to_numpy(svg_string):
    try:
        png_data = cairosvg.svg2png(bytestring=svg_string.encode('utf-8'), output_width=256, output_height=256, background_color="white")
        return np.array(Image.open(io.BytesIO(png_data)).convert('L'))
    except: return None

def select_best_svg(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = extract_svg(raw_svg)
        
        # INVESTIGATION PRINT
        if not is_kaggle_compliant(clean_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(clean_svg)}")
            print(f"Raw string: {clean_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(clean_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]

def select_best_svg_healed(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = heal_svg(extract_svg(raw_svg))

        # INVESTIGATION PRINT
        if not is_kaggle_compliant(clean_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(clean_svg)}")
            print(f"Raw string: {clean_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(clean_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]

The below process took 1687 minutes for 1000 samples.

In [ ]:
# ==========================================
# INFERENCE LOOP
# ==========================================
if __name__ == "__main__":
    test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv") 
    csv_filename = "./submission-3b-2048.csv"
    resume_index = 0
    
    # We will generate 2 candidates per prompt to feed to SigLIP
    NUM_CANDIDATES = 2

    MAX_CONTEXT_WINDOW = 2048
    
    print(f"\nStarting Inference on {len(test_df)} prompts...")

    for idx in range(resume_index, len(test_df)):
        row = test_df.iloc[idx]
        print(f"\n[{idx+1}/{len(test_df)}] Generating ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        open_comment = "<!-" + "-"
        close_comment = "-" + "->"
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        
        
        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        candidates = [] 
        
        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # We ask outlines to generate exactly NUM_CANDIDATES sequence at a time
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
                candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        best_svg = select_best_svg(row["prompt"], candidates)
        print(best_svg[:200])  # Print the first 200 chars of the best SVG for sanity check
        
        result_df = pd.DataFrame([{"id": row["id"], "svg": best_svg}])
        if idx == 0 or not os.path.exists(csv_filename):
            result_df.to_csv(csv_filename, index=False, mode='w')
        else:
            result_df.to_csv(csv_filename, index=False, mode='a', header=False)
            
        # Clear cache before moving to the next totally new prompt
        torch.cuda.empty_cache()

    print("\nLocal inference complete! Check submission-3b-2048.csv")


Starting Inference on 1000 prompts...

[1/1000] Generating ID: fa1d8fa7-080f-4269-a9cf-a17562c9a0ca | Prompt: firewood stack cut logs wood with leaf i...
  ~ Prompt tokens: 17 | Tokens available for SVG: 2021
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 2149
Raw string: 9,33.4 C2.9,33.5 2.9,33.6 2.9,33.7 C2.9,33.8 2.9,33.9 2.9,34 C2.9,34.1 2.9,34.2 2.9,34.3 C2.9,34.4 2
<svg xmlns="http://www.w3.org/2000/svg" height="200" viewBox="0 0 16 16" width="200"><path d="M13.5,12.5 C11.5,12.5 10,11 10,9 C10,9.5 10,10 10,10.5 C10,11.5 9.5,12.5 8.5,12.5 C9.5,11.5 10,10.5 10,10 

[2/1000] Generating ID: 6eede943219547c22ac56085027d33cc | Prompt: The image shows five horizontal lines of...
  ~ Prompt tokens: 27 | Tokens available for SVG: 2011
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" width="200"><pat

Forgot to note the time, but around the same amount of time as the per 2048 x2 attempt per sample.

Approximately, 545 Minutes for 323 samples.

In [ ]:
import pandas as pd
import warnings

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from Pass 1
sub_df = pd.read_csv("./submission-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 1  # Reduced due to Increased MAX_CONTEXT_WINDOW
MAX_CONTEXT_WINDOW = 4096  # We can also try to push this up since we know these are the hard cases

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_rescued-3b-2048.csv", index=False) 
else:
    # The 4096-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP 
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 4096 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed 
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_rescued-3b-2048.csv", index=False)

        # Clear cache before moving to the next totally new prompt
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_rescued-3b-2048.csv' is ready.")

Found 323 blank SVGs. Starting Rescue Run.

[1/323] Rescuing ID: 9e831fb6831745f4d15156b7a95e4f92 | Prompt: An orange hexagon with the number '13' i...
  ~ Prompt tokens: 21 | Tokens available for SVG: 4065
  ~ Generating 1 Candidates...
  ~ Judging 1 successful candidates with SigLIP...
.5,62.5 ZM87.5,137.5 L87.5,125 L68.75,125 L68.75,137.5 L87.5,137.5 ZM112.5,137.5 L112.5,125 L131.25,125 L131.25,137.5 L112.5,137.5 ZM112.5,62.5 L112.5,50 L131.25,50 L131.25,62.5 L112.5,62.5 Z"/></svg>

[2/323] Rescuing ID: 625181a2-600d-4db5-83c6-1a34676eb0dc | Prompt: The image is a black and white represent...
  ~ Prompt tokens: 26 | Tokens available for SVG: 4060
  ~ Generating 1 Candidates...
  ~ Judging 1 successful candidates with SigLIP...
,24.8 37.5,24.8 C39.2,24.8 40.5,26.1 40.5,27.8 Z"/><path d="M13.8,27.8 C13.8,29.5 12.5,30.8 10.8,30.8 C9.1,30.8 7.8,29.5 7.8,27.8 C7.8,26.1 9.1,24.8 10.8,24.8 C12.5,24.8 13.8,26.1 13.8,27.8 Z"/></svg>

[3/323] Rescuing ID: 8ba1bd7c-211c-43b0-a4aa-0e6e33c92486 

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import pandas as pd
import warnings
import torch
import gc # <--- CRITICAL: Import the garbage collector

# 1. Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# 2. Identify the missing rows from Pass 1
sub_df = pd.read_csv("./submission_rescued-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 1  # Reduced due to Increased MAX_CONTEXT_WINDOW
MAX_CONTEXT_WINDOW = 4096  # We can also try to push this up since we know these are the hard cases

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_rescued-3b-2048.csv", index=False) 
else:
    # The 4096-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP 
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 4096 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed 
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_rescued-3b-2048.csv", index=False)

        # Clear cache before moving to the next totally new prompt
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_rescued-3b-2048.csv' is ready.")

Found 239 blank SVGs. Starting Rescue Run.

[1/239] Rescuing ID: 7e63b44725fd31b7ee98d8ba88fe29cd | Prompt: A simple line drawing of a pencil with a...
  ~ Prompt tokens: 22 | Tokens available for SVG: 4064
  ~ Generating 1 Candidates...
  ~ Judging 1 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 4376
Raw string: L240.6,4.28 L241.4,4.36 L242.2,4.44 L243,4.52 L243.8,4.6 L244.6,4.68 L245.4,4.76 L246.2,4.84 L247,4.
      [X] ALL CANDIDATES FAILED. Returning empty fallback.
<svg></svg>

[2/239] Rescuing ID: 9bda86e2078baf4d4fa15be8de3bcdce | Prompt: A black outline of a smartphone icon wit...
  ~ Prompt tokens: 35 | Tokens available for SVG: 4051
  ~ Generating 1 Candidates...
  ~ Judging 1 successful candidates with SigLIP...
112.5 A12.5 12.5 0 1 1 75,137.5 A12.5 12.5 0 0 1 75,112.5 ZM62.5,112.5 A12.5 12.5 0 1 1 62.5,137.5 A12.5 12.5 0 0 1 62.5,112.5 ZM50,112.5 A12.5 12.5 0 1 1 50,137.5 A12.5 12.5 0 0 1 50,112.5 Z"/></svg>

[3/239] Rescuing ID: 1bb9

In [6]:
# Valid SVG checks

# 1. Identify the rows with char > 16000 (This includes blanks and any other invalid outputs that slipped through)
sub_df['svg'] = sub_df['svg'].fillna("<svg></svg>")
oversized_mask = sub_df['svg'].str.len() > 16000
oversized_ids = sub_df[oversized_mask]['id'].tolist()
fix_df = test_df[test_df['id'].isin(oversized_ids)]

print(f"Found {len(fix_df)} SVGs exceeding 16,000 characters.")

# 2. Identify the blank SVGs
missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 0 SVGs exceeding 16,000 characters.
Found 91 blank SVGs.


The below process took 237 Minutes, 12 seconds for 67 Samples

So, originally it should be 322 Minutes for 91 Samples. (Ran into a memory error on the 34th sample).

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from Pass 1
sub_df = pd.read_csv("./submission_rescued-3b-2048.csv")
# sub_df = pd.read_csv("./submission_fixed-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's a second pass
MAX_CONTEXT_WINDOW = 4096  # We can also try to push this up since we know these are the hard cases

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False) 
else:
    # The 4096-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP 
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 4096 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )

            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed from Pass 1 
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048.csv' is ready.")

Found 67 blank SVGs. Starting Rescue Run.

[1/67] Rescuing ID: 4b3c3ff3dcac0299c4c7c83307487357 | Prompt: An abstract black-and-white illustration...
  ~ Prompt tokens: 74 | Tokens available for SVG: 4012
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
th fill="currentColor" d="M126.5,121.2 L126.5,146.5 L188.9,146.5 L188.9,121.2 L126.5,121.2 Z"/><path fill="currentColor" d="M181.2,121.2 L181.2,146.5 L168.2,146.5 L168.2,121.2 L181.2,121.2 Z"/>
</svg>

[2/67] Rescuing ID: 92bc76fd445cfb23162ecfb03f75c2a5 | Prompt: A stylized icon featuring a curved line ...
  ~ Prompt tokens: 22 | Tokens available for SVG: 4064
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
.42 L163.88,67.85 A41.62 41.62 0 0 0 157.59,0 L42.62,0 A41.62 41.62 0 0 0 36.33,67.85 L36.33,94.42 L36.33,105.58 A41.62 41.62 0 0 0 42.62,160.84 L163.88,160.84 L163.88,170.84 L157.59,170.84 Z"/></svg>

[3/67] Rescuing ID: a4aaa1dea134b7ad933bc58f45a8abd7 | Prompt: A 

In [8]:
# Valid SVG checks

# 1. Identify the rows with char > 16000 (This includes blanks and any other invalid outputs that slipped through)
sub_df['svg'] = sub_df['svg'].fillna("<svg></svg>")
oversized_mask = sub_df['svg'].str.len() > 16000
oversized_ids = sub_df[oversized_mask]['id'].tolist()
fix_df = test_df[test_df['id'].isin(oversized_ids)]

print(f"Found {len(fix_df)} SVGs exceeding 16,000 characters.")

# 2. Identify the blank SVGs
missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 0 SVGs exceeding 16,000 characters.
Found 15 blank SVGs.


The below process took 15 Minutes and 15 seconds for 15 samples.

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from Pass 1
sub_df = pd.read_csv("./submission_fixed-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  
MAX_CONTEXT_WINDOW = 1568  # We can also try to push this up since we know these are the hard cases

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False) 
else:
    # The 1568-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP 
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped down to 1568 to see if limiting the generation length helps with compliance
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed 
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048.csv' is ready.")

Found 15 blank SVGs. Starting Rescue Run.

[1/15] Rescuing ID: 2bc39245-d640-43d8-9530-b286f936c0ba | Prompt: A shooting star with a bright tail and s...
  ~ Prompt tokens: 22 | Tokens available for SVG: 1536
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 1640
Raw string: 8,36.75 0.71,36.74 0.6,36.74 C0.51,36.74 0.42,36.74 0.33,36.74 C0.24,36.74 0.15,36.74 0.07,36.74 C0,
      [!] Candidate 2 failed compliance. Length: 1654
Raw string: 3 C15.81,14.32 15.98,14.59 16.15,14.88 C16.32,15.16 16.49,15.43 16.66,15.72 C16.83,15.99 17.0,16.26 
      [X] ALL CANDIDATES FAILED. Returning empty fallback.
<svg></svg>

[2/15] Rescuing ID: 45cc90ae5276bda487c4d20efc9f681c | Prompt: A simple gray wave-like shape on a plain...
  ~ Prompt tokens: 18 | Tokens available for SVG: 1540
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
152.97 120.64,180.93 100,180.93 ZM100,29.97 C100,29.97 73.8

In [11]:
# Valid SVG checks

# 1. Identify the rows with char > 16000 (This includes blanks and any other invalid outputs that slipped through)
sub_df['svg'] = sub_df['svg'].fillna("<svg></svg>")
oversized_mask = sub_df['svg'].str.len() > 16000
oversized_ids = sub_df[oversized_mask]['id'].tolist()
fix_df = test_df[test_df['id'].isin(oversized_ids)]

print(f"Found {len(fix_df)} SVGs exceeding 16,000 characters.")

# 2. Identify the blank SVGs
missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 0 SVGs exceeding 16,000 characters.
Found 6 blank SVGs.


The below process took 89 Minutes and 3 seconds for 6 Samples.

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows
sub_df = pd.read_csv("./submission_fixed-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's a second pass
MAX_CONTEXT_WINDOW = 8192  # We can also try to push this up since we know these are the hard cases

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False) 
else:
    # The 8192-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 8192 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048.csv' is ready.")

Found 6 blank SVGs. Starting Rescue Run.

[1/6] Rescuing ID: 2bc39245-d640-43d8-9530-b286f936c0ba | Prompt: A shooting star with a bright tail and s...
  ~ Prompt tokens: 22 | Tokens available for SVG: 8160
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 8782
Raw string: .11,541.02 L2213.34,542.44 L2219.57,543.86 L2225.8,545.28 L2232.03,546.7 L2238.26,548.12 L2244.49,54
      [!] Candidate 2 failed compliance. Length: 8887
Raw string: 2.75,18.75 L-112.75,18.25 L-112.75,17.25 L-112.75,17.75 L-113.5,18.75 L-113.5,18.25 L-113.5,17.25 L-
      [X] ALL CANDIDATES FAILED. Returning empty fallback.
<svg></svg>

[2/6] Rescuing ID: 112ba24ae9e0c6f096b9128625f8832c | Prompt: A black arrow icon with a bold downward ...
  ~ Prompt tokens: 31 | Tokens available for SVG: 8151
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 8785
Raw stri

The below process took 6 Minutes and 27 seconds for 4 Samples.

In [ ]:
import pandas as pd
import warnings
import torch
import gc # <--- CRITICAL: Import the garbage collector

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows 
sub_df = pd.read_csv("./submission_fixed-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's a second pass
MAX_CONTEXT_WINDOW = 2048  # We can try this again to catch the examples that failed the first time with 2048

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False) 
else:
    # The 2048-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 5100 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048.csv' is ready.")

Found 4 blank SVGs. Starting Rescue Run.

[1/4] Rescuing ID: 2bc39245-d640-43d8-9530-b286f936c0ba | Prompt: A shooting star with a bright tail and s...
  ~ Prompt tokens: 22 | Tokens available for SVG: 2016
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 2 failed compliance. Length: 2122
Raw string: C135.99,140.47 134.58,141.5 134.58,143.58 C134.58,145.99 136.36,148.8 139.14,149.8 C141.8,150.8 144.
33 100,105.94 M100,147.71 C100,158.92 110.37,169.28 124.27,171.3 C138.16,173.22 151.29,162.95 153.31,148.05 C155.33,133.15 145.06,122.78 131.16,120.76 C117.27,118.74 104.14,128.99 100,147.71"/>
</svg>

[2/4] Rescuing ID: 388ccc7936d31aaba4cecbff344ceb78 | Prompt: A minimalist spiral design with concentr...
  ~ Prompt tokens: 21 | Tokens available for SVG: 2017
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 2126
Raw string: .97,136.13 26.45,137.45 24.14,138

The below process took 2 Minutes and 15 Seconds for 2 samples.

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows 
# sub_df = pd.read_csv("./submission_rescued-3b-2048.csv")
sub_df = pd.read_csv("./submission_fixed-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's a second pass
MAX_CONTEXT_WINDOW = 1568  # Let's try 1568 tokens again. We know 2048 was too much for some examples, but maybe 1568 will be the sweet spot for the remaining ones.
# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False) 
else:
    # The 1568-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped down to 1568
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the select_best_svg_healed function once again
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048.csv' is ready.")

Found 2 blank SVGs. Starting Rescue Run.

[1/2] Rescuing ID: 388ccc7936d31aaba4cecbff344ceb78 | Prompt: A minimalist spiral design with concentr...
  ~ Prompt tokens: 21 | Tokens available for SVG: 1537
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 2 failed compliance. Length: 1661
Raw string: 213.75,195.13 L213.75,200.87 L213.75,200.87 C216.65,203.37 216.65,203.37 219.54,205.88 L219.54,205.8
130 C137.91,130 166.67,101.23 166.67,73.33 C166.67,45.44 137.91,16.67 100,16.67 ZM177.88,100 C177.88,138.9 145.44,171.25 107.58,177.78 L107.58,170.58 C145.44,177.08 177.88,199.42 177.88,100 Z"/></svg>

[2/2] Rescuing ID: 52b768babdfab2dde3b7f5754ab0356d | Prompt: The image features a black cloud partial...
  ~ Prompt tokens: 31 | Tokens available for SVG: 1527
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 1629
Raw string: 95.02,15.65 94.51,15.93 C93.97,16.19 

The below process took 5 Minutes and 5 seconds for 1 sample.

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows
# sub_df = pd.read_csv("./submission_rescued-3b-2048.csv")
sub_df = pd.read_csv("./submission_fixed-3b-2048.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's a second pass
MAX_CONTEXT_WINDOW = 4096  # Let's try 4096 tokens again to see if it can handle the longest prompts without guillotining (To prevent hallucinations while still giving more room than 1024)
# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found!")
    sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False) 
else:
    # The 4096-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 4096 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_fixed-3b-2048.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048.csv' is ready.")

Found 1 blank SVGs. Starting Rescue Run.

[1/1] Rescuing ID: 52b768babdfab2dde3b7f5754ab0356d | Prompt: The image features a black cloud partial...
  ~ Prompt tokens: 31 | Tokens available for SVG: 4055
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 4207
Raw string: 8.03,63.28 77.91,63.28 77.66,63.28 C77.48,63.28 77.19,63.28 76.99,63.28 C76.72,63.12 76.52,63.12 76.
133.33 A16.67 16.67 0 0 1 108.33,116.67 ZM100,116.67 A8.33 8.33 0 1 0 116.67,133.33 A8.33 8.33 0 0 0 100,116.67 ZM116.67,133.33 A8.33 8.33 0 1 0 133.33,116.67 A8.33 8.33 0 0 0 116.67,133.33 Z"/></svg>

Rescue complete! Final 'submission_fixed-3b-2048.csv' is ready.
